# 01 - Exploratory Data Analysis
struktur data, kualitas data, distribusi target, dan hubungan fitur dengan tingkat kerusakan sebelum modeling.

In [1]:
import sys
import warnings
warnings.filterwarnings("ignore")
from pathlib import Path

sys.path.insert(0, "..")      # agar folder scripts/ bisa di-import dari notebooks/

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scripts.preprocess import load_data, split_xy
from scripts.feature_engineering import (
    BASE_NUMERIC_COLS, GEO_COLS, SUPERSTRUCTURE_COL, split_materials,
)
from scripts.visualize import (
    plot_age_vs_damage, plot_categorical_vs_damage, plot_correlation_heatmap,
    plot_damage_by_district, plot_numeric_distributions, plot_superstructure_vs_damage,
    plot_target_distribution,
)

ON_KAGGLE = Path("/kaggle/input").exists()
if ON_KAGGLE:
    DATA_DIR = next(Path("/kaggle/input").glob("**/train.csv")).parent
    OUT_DIR = Path("/kaggle/working")
else:                                   
    DATA_DIR = Path("../data")
    OUT_DIR = Path("..")
print("DATA_DIR:", DATA_DIR, "| OUT_DIR:", OUT_DIR)

DATA_DIR: ..\data | OUT_DIR: ..


## 1. Memuat data

In [ ]:
train, test, sample = load_data(DATA_DIR)
print("train:", train.shape, "| test:", test.shape, "| sample:", sample.shape)
X, y = split_xy(train)
train.head()

## 2. Tipe data, nilai kosong, dan jumlah nilai unik

In [ ]:
pd.DataFrame({
    "dtype": train.dtypes.astype(str),
    "n_missing": train.isna().sum(),
    "pct_missing": (train.isna().mean() * 100).round(2),
    "n_unique": train.nunique(),
})

## 3. Distribusi target (`damage_grade`)
Ketidakseimbangan kelas menentukan strategi: Macro F1 memberi bobot sama untuk tiap kelas.

In [ ]:
counts = y.value_counts().sort_index()
display(pd.DataFrame({"jumlah": counts, "persen": (counts / counts.sum() * 100).round(2)}))
plot_target_distribution(y)

## 4. Fitur numerik
Cek outlier.

In [ ]:
display(train[BASE_NUMERIC_COLS].describe(percentiles=[.01, .5, .99]).T)
print("Korelasi Spearman dengan damage_grade:")
print(train[BASE_NUMERIC_COLS].assign(damage_grade=y.values)
      .corr(method="spearman")["damage_grade"].drop("damage_grade").round(3))
print("\nNilai umur terbanyak (cek kode khusus, mis. 999):")
print(train["age_building"].value_counts().head(8))
plot_numeric_distributions(train)
plot_age_vs_damage(train, y)

## 5. Fitur kategorikal vs tingkat kerusakan

In [ ]:
plot_categorical_vs_damage(train, y)

## 6. Kolom `superstructure`
Cek apakah satu sel berisi beberapa material.

In [ ]:
print(train[SUPERSTRUCTURE_COL].value_counts(dropna=False).head(15))
n_tok = train[SUPERSTRUCTURE_COL].fillna("unknown").astype(str).map(split_materials).map(len)
print("\nRata-rata material per bangunan:", n_tok.mean().round(2))
plot_superstructure_vs_damage(train, y)

## 7. Fitur geografis

In [ ]:
print(train[GEO_COLS].nunique())
print("Id wilayah di test yang tidak ada di train:",
      {c: int((~test[c].isin(train[c].unique())).sum()) for c in GEO_COLS})
plot_damage_by_district(train, y)

## 8. Korelasi fitur numerik

In [ ]:
plot_correlation_heatmap(X, y)

# 9. Temuan EDA
## Distribusi Kelas
- Total data: **533.465 baris**
- Dataset **tidak seimbang**
  - Mayoritas: **grade 5**
  - Minoritas: **grade 1**
- Pada OOF:
  - Sedikit **overpredict** kelas 2 dan 3
  - **Underpredict** kelas 5

---

## Fitur Numerik: Outlier dan Kode Khusus

| Fitur | Min | Max | Persentil 99% | Outlier | Catatan |
|---|---:|---:|---:|---|---|
| `count_floors_pre_eq` | 1 | 9 | 4 | >4 (5–9) | Histogram spike di 1, 2, 3 |
| `age_building` | 0 | 999 | 90 | >90 | **999 adalah kode khusus** (kemungkinan missing/unknown). Nilai terbanyak 15 dan 20. Distribusi right-skewed |
| `plinth_area_sq_ft` | 70 | 5000 | 1222 | >1222 | Right-skewed |
| `height_ft_pre_eq` | 6 | 99 | 32 | >32 | **99 kemungkinan kode khusus** (missing) |

---

## Kategori yang Paling Membedakan Tingkat Kerusakan
- Kategori paling kuat: **`superstructure`** (material struktur)

---

## Format `superstructure`
- Format: **one-hot encoding**
- Kolom: `has_superstructure_<material>` dengan nilai **0/1**

---

## Kardinalitas Fitur Geografis dan Risiko ID Baru di Test

| Fitur | Jumlah Unique |
|---|---:|
| `district_id` | 11 |
| `vdcmun_id` | 110 |
| `ward_id` | 945 |

- ID wilayah di test yang tidak ada di train:
  - `district_id`: **0**
  - `vdcmun_id`: **0**
  - `ward_id`: **0**
- Artinya: **tidak ada kategori baru di test**, sehingga risiko ID baru **rendah**.
- Namun, `ward_id` berkardinalitas tinggi (**945**), sehingga one-hot encoding akan terlalu **sparse** dan rawan **overfit**.
- Perlu encoding seperti **target encoding / frequency encoding** dengan **smoothing** dan **cross-fitting**.

---

## Keputusan Preprocessing

### a. Kode Khusus
- `age_building` = **999** → NaN / imputasi
- `height_ft_pre_eq` = **99** → NaN / imputasi
- `count_floors_pre_eq` > **4** → bisa di-cap di **4**

### b. Outlier
- `plinth_area_sq_ft` → cap pada persentil 99 (**1222**) atau log transform
- `height_ft_pre_eq` → cap pada **32**
- `age_building` → cap pada **90** setelah 999 diubah menjadi NaN

### c. Transformasi Numerik
- `age_log = log1p(age_building)` setelah handling 999
- `area_per_floor = plinth_area_sq_ft / count_floors_pre_eq`
- `height_per_floor = height_ft_pre_eq / count_floors_pre_eq`
- `area_to_height = plinth_area_sq_ft / height_ft_pre_eq`
- Scaling dengan **RobustScaler / StandardScaler** untuk reglog

### d. Encoding Kategorikal
- **One-hot encoding** untuk kardinalitas rendah:
  - `land_surface_condition`
  - `foundation_type`
  - `roof_type`
  - `ground_floor_type`
  - `other_floor_type`
  - `position`
  - `plan_configuration`
  - material `superstructure`
- Untuk `district_id`, `vdcmun_id`, `ward_id`:
  - **Target encoding / frequency encoding** dengan **smoothing + cross-fitting**

### e. Superstructure
- Konversi one-hot ke satu kolom kategorikal `material` (pilih yang bernilai 1)
- Atau tetap one-hot
- Jika multiple, buat kategori **`mixed`**

### f. Class Imbalance
- Gunakan `class_weight` seperti:
  - `[0.5, 0.8, 1.0, 1.1, 1.2]`
  - atau `balanced`
- Model final: **reglog dengan bobot kelas**

### g. Multikolinearitas
- `count_floors_pre_eq` dan `height_ft_pre_eq` berkorelasi **0.71**
  - Bisa drop salah satu atau gunakan regularisasi L2 (C tuning)
- `age_log` redundan dengan `age_building`
  - Pilih salah satu (`age_log` lebih normal)

### h. Seleksi Fitur
Fitur paling berpengaruh:
- `district_id`
- `ward_id`
- `vdcmun_id`
- `superstructure`
- `age_building`
- `height_ft_pre_eq`
- `plinth_area_sq_ft`
- `foundation_type`
- `roof_type`
- `ground_floor_type`

Fitur lain tetap dipakai dengan **regularisasi**.